# B.1 模型怎麼要求程式幫忙？


「123乘45」可以寫成一份工具請求：要用的工具叫 multiply，參數 a＝123、b＝45，答案先空著，等工具算。本單元說「工具的參數」時，指交給工具的輸入值（JSON 裡的 `arguments`），和模型裡可學的參數（parameter）是兩回事。只在回答裡寫「已使用計算器」沒有任何執行效果；模型要生成外層程式讀得懂的名稱與參數，再由外層真正運算。人看的時候常簡寫成 `multiply(a=123,b=45)`，但模型實際要生成的是下面這種 JSON 文字。**JSON** 是用文字寫資料的通用格式：大括號包住一組組「欄位名: 值」，寫法和 Python 字典很像，但它是一段字串，可以在程式之間傳遞。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import json

request = {"name": "multiply", "arguments": {"a": 123, "b": 45}}
training_example = {"user": "123乘45", "assistant_tool_request": request}
text = json.dumps(request)
print("訓練例子", training_example)
print("可傳遞的請求文字", text)
print("目前只有請求，尚未執行乘法")

`request` 是人工寫的字典，有 `name` 與 `arguments` 兩個欄位。`json.dumps` 把字典轉成 JSON 文字：印出來時欄位名一律用雙引號，這段文字才是模型要生成、外層要讀的東西。`training_example` 只是展示一筆示範資料的樣子，程式沒有訓練，也沒有呼叫工具。JSON 只是資料格式，寫出它不會讓 Python 自動執行乘法。

請求適合精確運算；問「乘法是什麼」則需要概念回答。兩種都示範，模型才學得到什麼時候該用工具。若把 45 錯寫成 54，外層照樣能合法算出 6642，卻不是原題的答案。留下原題和模型生成的原始請求，格式錯、選錯工具、參數錯才分得出來。

本單元的**工具實驗**另外從頭訓練了一個小模型，只學兩種題目：`CALC:add(2,3)` 這種 0 到 9 的兩數加法或乘法，要先提出工具請求；`COPY:3` 則是照抄數字，要直接回答 `{"done":true,"answer":3}`，不呼叫工具。它沒有學過正文的 123×45，也沒有學過一般中文問句。B.5 之後的**選卡實驗**是另一個模型，只學挑下一步的動作；兩個模型沒有串成同一個助理。

練習把 `request` 的 `"b": 45` 改成 `"b": 54`，先預測：JSON 文字照樣合法，只有參數變了；外層之後會算出 6642，卻不是原題的答案。這種錯，只有對照原題才抓得到。

<details>
<summary>補充：工具實驗的資料與重做方式</summary>

工具實驗的模型從隨機的參數開始，只用示範資料學會動作格式，沒有載入其他已訓練模型的參數。訓練題是 `CALC:add(2,3)` 或 `CALC:multiply(2,3)` 這類 0 到 9 的兩數運算，另有 `COPY:3` 這類照抄數字、直接回答的題目。

同一對數字的加法、乘法與交換順序的題目，例如 2+3、3+2、2×3、3×2，算同一個家族（family，[5.12](https://birdhackor.github.io/tiny-perceptron-vlm/5.12.html)），整組只分到訓練、驗證或測試其中一邊；每個 COPY 的數字另成一個家族。這只測固定格式的 ASCII 指令與小數字。ASCII 是英文字母、數字和常見標點用的基本字元編碼；CALC、COPY 都沒有測中文的自然問句，也沒有測正文的 123×45。

每一步的請求，以及工具結果送回對話（叫回填，[B.3](https://birdhackor.github.io/tiny-perceptron-vlm/B.3.html) 會講）之後的回答，都由模型實際生成，沒有拿資料集的答案替它改正請求裡的 a、b。完整配方與原始結果在[實驗紀錄 tools.json](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/tools.json)，這是機器可讀的完整結果，逐題結果在 `results.samples`。要重做時，在已依 [W.1](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.1) 準備好環境的專案根目錄執行：

```bash
python scripts/course_experiments/run.py --experiment tools --device cpu
```

這個指令用 CPU，輸出存在 `outputs/course-experiments/course-v1/tools/`。實驗紀錄裡的成績是在 NVIDIA L4 顯示卡上跑出來的；網站上的短程式沒有重跑這個實驗。

</details>
